In [29]:
!pip install -q -U google-genai pypdf

from google import genai
from google.genai import types
from google.colab import userdata
import numpy as np

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
print("Setup done.")

Setup done.


In [30]:
from google.colab import files
uploaded = files.upload()

pdf_name = list(uploaded.keys())[0]
print("Is uploaded?", pdf_name)

Saving Python_Notes.pdf to Python_Notes (2).pdf
Is uploaded? Python_Notes (2).pdf


In [31]:
from pypdf import PdfReader

reader = PdfReader(pdf_name)

print("Number of pages:", len(reader.pages))

text = ""
for page in reader.pages:
    text += page.extract_text() + "\n"

print("total characters:", len(text))
print()

print()
print("first few character:", text[:500])

Number of pages: 43
total characters: 22484


first few character:  
 
Introduction to Python: 
➢ Python is a high-level, interpreted, and general-purpose programming 
language. It was created by Guido van Rossum and first released in 1991. 
Python is known for its simple and readable syntax, which makes it an 
excellent choice for beginners as well as professionals. 
Key Features: 
➢ Easy to Learn and Use: 
 Python has a clear and straightforward syntax that is similar to 
English. 
➢ Interpreted Language: 
 Python code is executed line by line, which makes 


In [32]:
def chunk_text(text, chunk_size=800, overlap=150):
  chunks = []
  start = 0
  while start < len(text):
    end = start + chunk_size
    chunks.append(text[start:end])
    start = end - overlap
  return chunks

chunks = chunk_text(text)
print("total no of chunks", len(chunks))

total no of chunks 35


In [33]:
print("---------last 150 charcter of first chunk")
print(chunks[0][-150:])

print("---------first 150 charcter of second chunk")
print(chunks[1][:150])

---------last 150 charcter of first chunk
: 
 Python supports object-oriented programming (OOP), allowing 
concepts like classes and objects. 
PYTHON 
 
➢ Extensive Libraries: 
 Python has a
---------first 150 charcter of second chunk
: 
 Python supports object-oriented programming (OOP), allowing 
concepts like classes and objects. 
PYTHON 
 
➢ Extensive Libraries: 
 Python has a


In [34]:
EMP_MODEL = "gemini-embedding-001"

def embed_chunks(t):
  result = client.models.embed_content(
  model = EMP_MODEL,
  contents=t,
  config=types.EmbedContentConfig(output_dimensionality=768)
  )
  return np.array(result.embeddings[0].values)


chunk_embeddings = []
for i, ch in enumerate(chunks):
  chunk_embeddings.append(embed_chunks(ch))
  print(f"embeeded chunks {i+1}/{len(chunks)}", end="\r")

chunk_embeddings = np.array(chunk_embeddings)
print()
print("shape", chunk_embeddings.shape)

embeeded chunks 35/35
shape (35, 768)


In [35]:
chunk_embeddings = chunk_embeddings / np.linalg.norm(
    chunk_embeddings, axis=1, keepdims=True
)
print("store", len(chunks), "chunks in vector store", chunk_embeddings.shape)

store 35 chunks in vector store (35, 768)


In [36]:
def retrieve(question, k=3):
  question_embedding = embed_chunks(question)
  question_embedding = question_embedding / np.linalg.norm(question_embedding)

  scores = chunk_embeddings @ question_embedding

  top_index = np.argsort(scores)[::-1][:k]

  return [(chunks[i], scores[i]) for i in top_index]

In [37]:
results = retrieve("Who created Python and when was it first released?")

for chunk, score in results:
  print(f"score: {score:.2f}")
  print(chunk)

score: 0.74
 
 
Introduction to Python: 
➢ Python is a high-level, interpreted, and general-purpose programming 
language. It was created by Guido van Rossum and first released in 1991. 
Python is known for its simple and readable syntax, which makes it an 
excellent choice for beginners as well as professionals. 
Key Features: 
➢ Easy to Learn and Use: 
 Python has a clear and straightforward syntax that is similar to 
English. 
➢ Interpreted Language: 
 Python code is executed line by line, which makes debugging easier. 
➢ Dynamically Typed: 
 You don't need to declare the data type of a variable; Python 
determines it at runtime. 
➢ Object-Oriented: 
 Python supports object-oriented programming (OOP), allowing 
concepts like classes and objects. 
PYTHON 
 
➢ Extensive Libraries: 
 Python has a
score: 0.66
nt("Hello, World!") 
✓ This simple program prints the message Hello, World!, showcasing Python's 
simplicity. 
 
Why is Python called "Python"? 
➢ The name "Python" was chosen

In [38]:
MODEL = "gemini-3.1-flash-lite"

Answer_configuration = types.GenerateContentConfig(
    system_instruction=(
    "You are an AI assistant that helps people find information inside only this document that is provided to you."
    "if the answer is not in the context, reply exactly:"
    "I could not find this in the document. Never make any guess with the user question."
    ),
    temperature=0.5
)


def ask(question, k=3):
  top = retrieve(question, k)

  content = "\n\n---------\n\n".join(chunk for chunk, score in top)
  prompt = f"""Context From the document:

{content}

QUESTION: {question}

Answer using only the context above. """

  repsonse = client.models.generate_content(

         model = MODEL,
         contents = prompt,
         config = Answer_configuration
    )

  return repsonse.text

In [39]:
print(ask("Who created Python and when was it first released?"))

Python was created by Guido van Rossum and first released in 1991.


In [44]:
print(ask("What is the result of 10 ** 3?"))

1000


In [51]:
print(ask("What does the pass statement do in Python?"))

The pass statement does nothing (placeholder).


In [52]:
print(ask("What is the latest stable version of Python?"))

I could not find this in the document.


In [53]:
def chunk_text(text, chunk_size=3000, overlap=150):
  chunks = []
  start = 0
  while start < len(text):
    end = start + chunk_size
    chunks.append(text[start:end])
    start = end - overlap
  return chunks

chunks = chunk_text(text)
print("total no of chunks", len(chunks))

total no of chunks 8


In [54]:
print("---------last 150 charcter of first chunk")
print(chunks[0][-150:])

print("---------first 150 charcter of second chunk")
print(chunks[1][:150])

---------last 150 charcter of first chunk
keywords (like if, class, for) cannot be used as variable names 
❖ Only underscores are allowed as special characters (no @, #, $, etc.) 
❖ Use meanin
---------first 150 charcter of second chunk
keywords (like if, class, for) cannot be used as variable names 
❖ Only underscores are allowed as special characters (no @, #, $, etc.) 
❖ Use meanin


In [55]:
chunk_embeddings = []

for i, ch in enumerate(chunks):
  chunk_embeddings.append(embed_chunks(ch))
  print(f"embeeded chunks {i+1}/{len(chunks)}", end="\r")

chunk_embeddings = np.array(chunk_embeddings)
print()
print("shape", chunk_embeddings.shape)

embeeded chunks 8/8
shape (8, 768)


In [56]:
chunk_embeddings = chunk_embeddings / np.linalg.norm(
    chunk_embeddings, axis=1, keepdims=True
)

print("store", len(chunks), "chunks in vector store", chunk_embeddings.shape)

store 8 chunks in vector store (8, 768)


In [63]:
results = retrieve("Who created Python and when was it first released?")

for chunk, score in results:
  print(f"score: {score:.2f}")
  print(chunk)

score: 0.71
 
 
Introduction to Python: 
➢ Python is a high-level, interpreted, and general-purpose programming 
language. It was created by Guido van Rossum and first released in 1991. 
Python is known for its simple and readable syntax, which makes it an 
excellent choice for beginners as well as professionals. 
Key Features: 
➢ Easy to Learn and Use: 
 Python has a clear and straightforward syntax that is similar to 
English. 
➢ Interpreted Language: 
 Python code is executed line by line, which makes debugging easier. 
➢ Dynamically Typed: 
 You don't need to declare the data type of a variable; Python 
determines it at runtime. 
➢ Object-Oriented: 
 Python supports object-oriented programming (OOP), allowing 
concepts like classes and objects. 
PYTHON 
 
➢ Extensive Libraries: 
 Python has a large standard library and many third-party libraries 
for tasks like web development, data science, machine learning, etc. 
➢ Cross-Platform: 
 Python programs can run on different oper

In [64]:
print(ask("Who created Python and when was it first released?"))

Python was created by Guido van Rossum and first released in 1991.


In [73]:
print(ask("What does the pass statement do in Python?"))

The pass statement does nothing (it is a placeholder).
